# Issue Salience and Anti-Elite Attitudes: ANES 2024

The CES batteries in notebooks 11-13 measure *positions* — where a respondent stands on immigration, healthcare, abortion — but not *salience*: which problem the respondent thinks matters most. A voter can hold a conservative immigration position and still vote on prices. The ANES 2024 Time Series asks the open-ended "most important problem" (MIP), a ten-issue closed-ended importance battery, which party would handle each issue better, and the anti-elite items (government run by a few big interests, corruption, strong-leader) that the CES lacks.

**Questions.** Among the 2024 switchers and new Trump voters, does immigration outrank the cost of living as the most important problem? Which party did they think would handle prices better? Are they distinguished by anti-elite / populist attitudes, or by cultural conservatism, or by economic views?

**Inputs:** `data/raw/anes/anes_2024_core.csv` (from `src/data_collection/anes_pull.py`; N = 5,521)

**Weights.** `V240107b` (post-election, full sample) throughout, because every salience item is post-election. Respondents without a post interview (557) drop. ANES is a much smaller sample than the CES; the Biden→Trump switcher group is **69 respondents** and the new-voter groups are ~170-200 each. Treat every switcher estimate as a wide interval.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')
RAW_DIR = Path('../data/raw'); TABLES_DIR = Path('../outputs/tables')
d = pd.read_csv(RAW_DIR / 'anes' / 'anes_2024_core.csv')
d = d[d['weight_post'].notna() & (d['weight_post'] > 0)].copy()
d['wt'] = d['weight_post']
def wmean(x, w):
    m = x.notna() & w.notna()
    return np.average(x[m], weights=w[m]) if m.any() else np.nan
def wshare(mask, w):
    return w[mask].sum() / w.sum()
def grp_table(df, cols, by='group', order=None):
    rows = {}
    for g, sub in df.groupby(by):
        rows[g] = {c: wmean(sub[c], sub['wt']) for c in cols}
        rows[g]['N'] = len(sub)
    t = pd.DataFrame(rows).T
    return t.loc[order] if order else t
print(len(d), 'respondents with post weight')

4964 respondents with post weight


## 1. Vote groups and a calibration check

In [2]:
d['group'] = np.select(
    [(d.vote20 == 'D') & (d.vote24 == 'D'), (d.vote20 == 'D') & (d.vote24 == 'R'),
     (d.vote20 == 'NV') & (d.vote24 == 'D'), (d.vote20 == 'NV') & (d.vote24 == 'R'),
     (d.vote20 == 'R') & (d.vote24 == 'R'), (d.vote20 == 'R') & (d.vote24 == 'D'),
     (d.vote20 == 'D') & (d.vote24 == 'NV'), (d.vote20 == 'R') & (d.vote24 == 'NV')],
    ['Biden->Harris', 'Biden->Trump', 'New->Harris', 'New->Trump', 'Trump->Trump', 'Trump->Harris', 'Biden->stay home', 'Trump->stay home'], None)
ORDER = ['Biden->Harris', 'Biden->Trump', 'New->Harris', 'New->Trump', 'Trump->Harris', 'Trump->Trump']
print(d['group'].value_counts())
v = d[d.vote24.isin(['D', 'R'])]
two_party = v.groupby('vote24')['wt'].sum() / v['wt'].sum()
print('\nANES 2024 two-party vote (post-weighted): Harris %.1f%%, Trump %.1f%%' % (100*two_party['D'], 100*two_party['R']))
print('Certified: Harris 49.2%, Trump 50.8% (two-party). ANES over-represents Harris voters by ~4 points, as it usually does for the winner\'s opponent in a partisan-response environment; group *comparisons* are the object here, not levels.')
b = d[d.vote20 == 'D']
print('\nAmong Biden-2020 voters: switch to Trump %.1f%% weighted (%d of %d); stayed home %.1f%%' % (
    100*wshare(b.vote24 == 'R', b.wt), (b.vote24 == 'R').sum(), len(b), 100*wshare(b.vote24 == 'NV', b.wt)))
t = d[d.vote20 == 'R']
print('Among Trump-2020 voters: switch to Harris %.1f%% (%d of %d); stayed home %.1f%%' % (
    100*wshare(t.vote24 == 'D', t.wt), (t.vote24 == 'D').sum(), len(t), 100*wshare(t.vote24 == 'NV', t.wt)))
print('CES 2024 cross-section (nb 11): Biden->Trump 5.2%, Trump->Biden 2.6%.')

group
Biden->Harris       1945
Trump->Trump        1437
New->Trump           198
New->Harris          166
Biden->stay home     142
Trump->stay home     101
Biden->Trump          69
Trump->Harris         58
Name: count, dtype: int64

ANES 2024 two-party vote (post-weighted): Harris 54.4%, Trump 45.6%
Certified: Harris 49.2%, Trump 50.8% (two-party). ANES over-represents Harris voters by ~4 points, as it usually does for the winner's opponent in a partisan-response environment; group *comparisons* are the object here, not levels.

Among Biden-2020 voters: switch to Trump 3.8% weighted (69 of 2245); stayed home 7.2%
Among Trump-2020 voters: switch to Harris 3.9% (58 of 1646); stayed home 8.1%
CES 2024 cross-section (nb 11): Biden->Trump 5.2%, Trump->Biden 2.6%.


## 2. The most important problem, by vote group

Open-ended, post-election, coded by ANES to Appendix B categories and grouped here into families (`MIP_FAMILY` in the pull script). The analytic MIP is the "single most important" follow-up when the respondent gave more than one problem, else the first mention.

In [3]:
m = d[d.mip_family.notna() & d.group.isin(ORDER)]
mip = pd.crosstab(m.group, m.mip_family, values=m.wt, aggfunc='sum', normalize='index').loc[ORDER]
mip = mip[mip.loc['Biden->Trump'].sort_values(ascending=False).index]
mip['N'] = m.groupby('group').size().loc[ORDER]
print('Share naming each family as the most important problem (row %):')
print((100*mip.drop(columns='N')).round(1).assign(N=mip['N']))
mip.to_csv(TABLES_DIR / 'anes_mip_by_group.csv')
# specific codes among switchers and new Trump voters
for g in ['Biden->Trump', 'New->Trump', 'Biden->Harris']:
    sub = m[m.group == g]
    top = sub.groupby('mip_code')['wt'].sum().sort_values(ascending=False).head(6) / sub.wt.sum()
    print(f'\n{g} top codes (41 immigration, 54 inflation, 50 economy, 48 politicians, 81 unity/division, 43 political corruption, 61 abortion, 78 elections):')
    print((100*top).round(1).to_dict())

Share naming each family as the most important problem (row %):
mip_family     economy_prices  economy_other  immigration  other  foreign_war  \
group                                                                           
Biden->Harris            17.7           12.0          3.3   10.2          2.3   
Biden->Trump             34.8           16.8         16.2   12.3          7.2   
New->Harris              21.5           16.2          5.8    9.4          3.0   
New->Trump               34.5           11.3         19.6    9.9          7.9   
Trump->Harris            26.7           20.3          7.6    5.6          4.7   
Trump->Trump             31.1            7.9         31.0    6.1          5.5   

mip_family     democracy_govt  culture_moral  crime_drugs  education  \
group                                                                  
Biden->Harris            30.2           10.5          2.7        1.7   
Biden->Trump              6.2            5.2          1.4        0.0   

In [4]:
# any-mention flags: did the respondent mention the family anywhere in up to three problems?
any_cols = ['mip_any_economy_prices', 'mip_any_economy_other', 'mip_any_immigration', 'mip_any_democracy_govt', 'mip_any_culture_moral']
anym = grp_table(m, any_cols, order=ORDER)
print('Share mentioning each family among up to three problems (%):')
print((100*anym.drop(columns='N')).round(1).assign(N=anym['N']))
anym.to_csv(TABLES_DIR / 'anes_mip_any_mention_by_group.csv')

Share mentioning each family among up to three problems (%):
               mip_any_economy_prices  mip_any_economy_other  \
Biden->Harris                    33.7                   37.5   
Biden->Trump                     59.9                   26.6   
New->Harris                      42.2                   42.3   
New->Trump                       57.4                   37.1   
Trump->Harris                    55.8                   46.9   
Trump->Trump                     65.5                   30.3   

               mip_any_immigration  mip_any_democracy_govt  \
Biden->Harris                 17.2                    51.0   
Biden->Trump                  36.4                    16.6   
New->Harris                   23.2                    29.0   
New->Trump                    39.2                    26.4   
Trump->Harris                 30.6                    36.2   
Trump->Trump                  63.3                    24.7   

               mip_any_culture_moral       N  
Biden->H

## 3. Closed-ended importance: immigration vs. the cost of living

Ten issues rated 1 (extremely important) to 5 (not at all). Reported as the share saying "extremely important" and as the direct comparison of illegal immigration against the cost of living within the same respondent.

In [5]:
imp_cols = {c: c.replace('imp_', '') for c in d.columns if c.startswith('imp_') and c != 'imp_immig_gt_cost'}
for c in imp_cols:
    d[c + '_ext'] = (d[c] == 1).astype(float).where(d[c].notna())
ext = grp_table(d[d.group.isin(ORDER)], [c + '_ext' for c in imp_cols], order=ORDER)
ext.columns = [c.replace('imp_', '').replace('_ext', '') for c in ext.columns]
print('Share rating the issue "extremely important" (%):')
print((100*ext.drop(columns='N')).round(1).assign(N=ext['N']))
ext.to_csv(TABLES_DIR / 'anes_importance_extremely_by_group.csv')
# head-to-head
cmp = d[d.group.isin(ORDER) & d.imp_immig_gt_cost.notna()]
hh = pd.crosstab(cmp.group, cmp.imp_immig_gt_cost, values=cmp.wt, aggfunc='sum', normalize='index').loc[ORDER]
hh.columns = ['cost of living more important', 'tie', 'illegal immigration more important']
print('\nWithin-respondent comparison (row %):')
print((100*hh).round(1))
hh.to_csv(TABLES_DIR / 'anes_immig_vs_cost_by_group.csv')

Share rating the issue "extremely important" (%):
               illegal_immig  cost_living  climate  guns  abortion  crime  \
Biden->Harris           19.7         53.7     56.2  48.5      48.9   29.8   
Biden->Trump            33.9         64.0     24.6  38.6      29.6   50.7   
New->Harris             21.6         65.2     48.6  43.5      47.5   34.0   
New->Trump              60.7         73.6     15.5  26.7      30.6   60.6   
Trump->Harris           36.3         59.9     39.7  40.0      37.2   55.7   
Trump->Trump            71.7         71.4      7.2  22.9      27.7   58.9   

               gaza  schools  democracy  racial_ineq       N  
Biden->Harris  20.0     41.3       77.1         46.3  1945.0  
Biden->Trump   13.1     34.1       33.3         23.4    69.0  
New->Harris    28.3     39.6       54.2         48.8   166.0  
New->Trump     20.4     54.0       42.9         20.1   198.0  
Trump->Harris  20.2     55.2       58.2         27.0    58.0  
Trump->Trump   18.0     61.1    

## 4. Issue ownership: which party would handle it better?

The test of the "conceded economic framing" hypothesis is here. If the switchers and new Trump voters thought Republicans would handle *prices* better, they switched on economics under a Republican frame; if they thought the parties equal on prices but Republicans better on immigration, they switched on immigration.

In [6]:
own_cols = [c for c in d.columns if c.startswith('own_') and c not in ('own_democracy', 'own_racial_ineq')]
rows = []
for g in ORDER:
    sub = d[d.group == g]
    for c in own_cols:
        s = sub[sub[c].notna()]
        rows.append({'group': g, 'issue': c.replace('own_', ''),
                     'Democrats': wshare(s[c] == 1, s.wt), 'Republicans': wshare(s[c] == 2, s.wt), 'no difference': wshare(s[c] == 3, s.wt), 'N': len(s)})
own = pd.DataFrame(rows)
own.to_csv(TABLES_DIR / 'anes_issue_ownership_by_group.csv', index=False)
for issue in ['cost_living', 'illegal_immig', 'abortion', 'crime']:  # democracy and racial_ineq ownership items were PAPI-only (N~200)
    t = own[own.issue == issue].set_index('group')[['Democrats', 'Republicans', 'no difference', 'N']]
    print(f'\nWhich party better on {issue} (row %):')
    print((100*t[['Democrats', 'Republicans', 'no difference']]).round(1).assign(N=t['N']))


Which party better on cost_living (row %):
               Democrats  Republicans  no difference     N
group                                                     
Biden->Harris       57.7          8.6           33.7  1939
Biden->Trump        10.9         72.7           16.4    68
New->Harris         51.8         13.9           34.3   166
New->Trump           2.1         83.1           14.8   197
Trump->Harris       19.1         48.5           32.5    57
Trump->Trump         1.6         91.1            7.3  1423

Which party better on illegal_immig (row %):
               Democrats  Republicans  no difference     N
group                                                     
Biden->Harris       49.0         17.8           33.2  1937
Biden->Trump         1.2         86.8           12.0    68
New->Harris         50.5         24.1           25.4   166
New->Trump           4.4         85.7            9.8   197
Trump->Harris       22.1         55.2           22.7    57
Trump->Trump         2.4 

In [7]:
# Party better on the respondent's OWN first-mentioned problem
mp = d[d.group.isin(ORDER) & d.mip1_party.notna()]
t = pd.crosstab(mp.group, mp.mip1_party, values=mp.wt, aggfunc='sum', normalize='index').loc[ORDER]
t.columns = ['Democrats', 'Republicans', 'no difference']
print('Party better able to handle the respondent\'s own most important problem (row %):')
print((100*t).round(1).assign(N=mp.groupby('group').size().loc[ORDER]))
t.to_csv(TABLES_DIR / 'anes_mip_party_by_group.csv')

Party better able to handle the respondent's own most important problem (row %):
               Democrats  Republicans  no difference     N
group                                                     
Biden->Harris       69.3          4.1           26.7  1922
Biden->Trump         3.0         69.7           27.4    68
New->Harris         54.9          5.6           39.5   162
New->Trump           2.6         83.6           13.8   197
Trump->Harris       38.8         27.7           33.5    57
Trump->Trump         1.1         90.5            8.4  1411


## 5. Anti-elite, populist, and cultural attitudes

Indices scaled 0-1 (1 = more of the named thing). Components verified against the codebook:

- **Anti-elite:** government run by a few big interests (V241231); how many in government corrupt (V241233, all=1); trust government never (V241229); outraged (V241120); public officials don't care what people like me think (V242200); people like me have no say (V242201); strong leader who bends rules (V242411); business leaders should run things (V242413); citizens in referendums (V242415).
- **Economic left:** services scale (V241239); government vs private insurance (V241245); guaranteed jobs/income (V241252); raise minimum wage (V242350); require paid leave (V241369x).
- **Immigration conservative:** wall (V241395x); send back Dreamers (V241392x); end birthright citizenship (V241389x); decrease immigration (V242227); illegal immigration raises crime (V242231x); oppose path to citizenship (V242234x).
- **Cultural conservative:** oppose trans bathroom access (V241372x); ban trans girls from sports (V241375x); oppose trans military service (V242364x); oppose campus DEI (V241288); diversity made US worse (V242349x); oppose gay marriage (V241385x); restrict abortion (V241302).

In [8]:
def sc(x, lo, hi):  # map lo..hi onto 0..1 with lo -> 0
    return (x - lo) / (hi - lo)
d['anti_elite'] = pd.concat([
    d.big_interests, sc(5 - d.govt_corrupt, 0, 4), sc(d.trust_govt, 1, 5), sc(d.outraged, 1, 5),
    sc(5 - d.officials_dont_care, 0, 4), sc(5 - d.no_say_in_govt, 0, 4), d.strong_leader_agree, sc(5 - d.cses_business_run, 0, 4), sc(5 - d.cses_referendums, 0, 4)], axis=1).mean(axis=1)
d['econ_left'] = pd.concat([
    sc(d.scale_services, 1, 7), sc(7 - d.scale_health_ins, 0, 6), sc(7 - d.scale_jobs_income, 0, 6),
    (d.min_wage == 1).astype(float).where(d.min_wage.notna()), sc(7 - d.paid_leave, 0, 6)], axis=1).mean(axis=1)
d['immig_cons'] = pd.concat([
    sc(7 - d.wall, 0, 6), sc(7 - d.dreamers, 0, 6), sc(7 - d.birthright_end, 0, 6),
    sc(d.immig_levels, 1, 5), sc(7 - d.immig_crime, 0, 6), sc(d.path_citizenship, 1, 7)], axis=1).mean(axis=1)
d['culture_cons'] = pd.concat([
    sc(d.trans_bathroom, 1, 7), sc(7 - d.trans_sports_ban, 0, 6), sc(d.trans_military, 1, 7),
    d.dei_campus.map({1: 0.0, 2: 1.0, 3: 0.5}), sc(d.diversity_better, 1, 5), sc(d.gay_marriage, 1, 7), sc(4 - d.abortion_std, 0, 3)], axis=1).mean(axis=1)
d['gov_health_ins'] = (d.scale_health_ins <= 3).astype(float).where(d.scale_health_ins.notna())
d['raise_min_wage'] = (d.min_wage == 1).astype(float).where(d.min_wage.notna())
d['therm_gap'] = d.therm_trump - d.therm_harris
d['officials_dont_care_agree'] = (d.officials_dont_care <= 2).astype(float).where(d.officials_dont_care.notna())
idx = ['anti_elite', 'econ_left', 'immig_cons', 'culture_cons', 'big_interests', 'officials_dont_care_agree', 'strong_leader_agree', 'gov_health_ins', 'raise_min_wage', 'therm_transgender', 'therm_gap']
att = grp_table(d[d.group.isin(ORDER)], idx, order=ORDER)
print(att.round(3))
att.to_csv(TABLES_DIR / 'anes_attitudes_by_group.csv')
print('\nSD-scaled gap, Biden->Trump minus Biden->Harris (in pooled SDs):')
bh, bt = d[d.group == 'Biden->Harris'], d[d.group == 'Biden->Trump']
for c in ['anti_elite', 'econ_left', 'immig_cons', 'culture_cons']:
    print(f'  {c:13s} {(wmean(bt[c], bt.wt) - wmean(bh[c], bh.wt)) / d[c].std():+.2f}')

               anti_elite  econ_left  immig_cons  culture_cons  big_interests  \
Biden->Harris       0.480      0.767       0.351         0.272          0.760   
Biden->Trump        0.616      0.588       0.561         0.538          0.948   
New->Harris         0.530      0.793       0.376         0.276          0.853   
New->Trump          0.585      0.557       0.613         0.585          0.816   
Trump->Harris       0.535      0.606       0.504         0.421          0.846   
Trump->Trump        0.623      0.411       0.689         0.670          0.915   

               officials_dont_care_agree  strong_leader_agree  gov_health_ins  \
Biden->Harris                      0.695                0.244           0.713   
Biden->Trump                       0.794                0.582           0.464   
New->Harris                        0.802                0.324           0.760   
New->Trump                         0.620                0.531           0.370   
Trump->Harris              

## 6. What predicts switching among Biden-2020 voters?

Weighted logit, HC1 SEs, on the four indices (standardized) plus the two salience flags. Seventy events; the point is the relative size of the coefficients, not significance.

In [9]:
b = d[(d.vote20 == 'D') & d.vote24.isin(['D', 'R'])].copy()
b['switch'] = (b.vote24 == 'R').astype(float)
b['mip_immig'] = (b.mip_family == 'immigration').astype(float).where(b.mip_family.notna())
b['mip_prices'] = (b.mip_family == 'economy_prices').astype(float).where(b.mip_family.notna())
b['rep_better_prices'] = (b.own_cost_living == 2).astype(float).where(b.own_cost_living.notna())
X = ['anti_elite', 'econ_left', 'immig_cons', 'culture_cons']
def fit(cols, label):
    sub = b.dropna(subset=cols + ['switch'])
    Z = sub[cols].copy()
    for c in X:
        if c in Z: Z[c] = (Z[c] - Z[c].mean()) / Z[c].std()
    Z = sm.add_constant(Z)
    r = sm.GLM(sub['switch'], Z, family=sm.families.Binomial(), freq_weights=sub['wt'] / sub['wt'].mean()).fit(cov_type='HC1')
    out = pd.DataFrame({'coef': r.params, 'se': r.bse, 'p': r.pvalues}).round(3)
    print(f'\n{label}: N={len(sub)}, events={int(sub.switch.sum())}')
    print(out)
    return out
m1 = fit(X, 'M1 attitudes only')
m2 = fit(X + ['mip_immig', 'mip_prices'], 'M2 + most important problem')
m3 = fit(X + ['mip_immig', 'mip_prices', 'rep_better_prices'], 'M3 + Republicans better on prices')
m4 = fit(['immig_cons', 'culture_cons'], 'M4 immigration vs culture only')
pd.concat({'M1': m1, 'M2': m2, 'M3': m3, 'M4': m4}).to_csv(TABLES_DIR / 'anes_switch_models.csv')


M1 attitudes only: N=1951, events=64
               coef     se      p
const        -4.518  0.205  0.000
anti_elite    0.807  0.158  0.000
econ_left    -0.390  0.121  0.001
immig_cons    0.472  0.153  0.002
culture_cons  0.863  0.169  0.000

M2 + most important problem: N=1925, events=62
               coef     se      p
const        -4.935  0.218  0.000
anti_elite    0.895  0.164  0.000
econ_left    -0.433  0.128  0.001
immig_cons    0.446  0.160  0.005
culture_cons  0.788  0.176  0.000
mip_immig     1.225  0.449  0.006
mip_prices    1.042  0.301  0.001

M3 + Republicans better on prices: N=1921, events=61
                    coef     se      p
const             -5.725  0.302  0.000
anti_elite         0.812  0.143  0.000
econ_left         -0.375  0.132  0.004
immig_cons         0.489  0.194  0.011
culture_cons       0.571  0.179  0.001
mip_immig          0.756  0.470  0.108
mip_prices         1.075  0.323  0.001
rep_better_prices  2.524  0.288  0.000

M4 immigration vs culture only: 

In [10]:
# Simple partition: switch rate by immigration-conservative (>=0.5) x MIP-is-immigration among Biden-2020 voters
b['imm_c'] = (b.immig_cons >= 0.5)
b['mip_is_immig'] = b.mip_immig == 1
part = b.dropna(subset=['mip_immig']).groupby(['imm_c', 'mip_is_immig']).apply(
    lambda s: pd.Series({'switch_rate': wmean(s.switch, s.wt), 'N': len(s), 'switchers': int(s.switch.sum())}))
print('Switch rate among Biden-2020 voters by immigration position x whether immigration is their MIP:')
print(part.round(3))
part.to_csv(TABLES_DIR / 'anes_switch_by_immig_position_x_salience.csv')
# share of switchers who are immigration-conservative (2024 attitudes) - comparison with CES 81%
sw = b[b.switch == 1]
print('\nShare of switchers immigration-conservative (index >= 0.5): %.0f%%; among Biden->Harris: %.0f%%' % (
    100*wshare(sw.imm_c, sw.wt), 100*wshare(b[b.switch == 0].imm_c, b[b.switch == 0].wt)))

Switch rate among Biden-2020 voters by immigration position x whether immigration is their MIP:
                    switch_rate       N  switchers
imm_c mip_is_immig                                
False False               0.018  1537.0       20.0
      True                0.156    40.0        3.0
True  False               0.125   317.0       31.0
      True                0.213    31.0        8.0

Share of switchers immigration-conservative (index >= 0.5): 60%; among Biden->Harris: 19%


## 7. New voters: salience and attitudes

In [11]:
nv = d[d.group.isin(['New->Harris', 'New->Trump'])].copy()
nv['trump'] = (nv.group == 'New->Trump').astype(float)
print('New voters 2024 (did not vote 2020): N Harris=%d, N Trump=%d; weighted Trump share %.1f%%' % (
    (nv.trump == 0).sum(), (nv.trump == 1).sum(), 100*wmean(nv.trump, nv.wt)))
print('CES 2024 new voters: Trump 58%. ANES younger/less-attached voters are under-recruited, so the ANES new-voter Trump share is not comparable in level.')
print('\nAge (mean): Harris %.1f, Trump %.1f' % (wmean(nv[nv.trump == 0].age, nv[nv.trump == 0].wt), wmean(nv[nv.trump == 1].age, nv[nv.trump == 1].wt)))
sub = nv.dropna(subset=X + ['trump'])
Z = sub[X].copy()
for c in X: Z[c] = (Z[c] - Z[c].mean()) / Z[c].std()
r = sm.GLM(sub['trump'], sm.add_constant(Z), family=sm.families.Binomial(), freq_weights=sub['wt'] / sub['wt'].mean()).fit(cov_type='HC1')
print('\nNew voters: Trump vs Harris on standardized indices (N=%d)' % len(sub))
print(pd.DataFrame({'coef': r.params, 'se': r.bse, 'p': r.pvalues}).round(3))

New voters 2024 (did not vote 2020): N Harris=166, N Trump=198; weighted Trump share 55.2%
CES 2024 new voters: Trump 58%. ANES younger/less-attached voters are under-recruited, so the ANES new-voter Trump share is not comparable in level.

Age (mean): Harris 35.2, Trump 36.2

New voters: Trump vs Harris on standardized indices (N=349)
               coef     se      p
const         0.646  0.180  0.000
anti_elite    0.288  0.180  0.110
econ_left    -0.827  0.196  0.000
immig_cons    0.766  0.230  0.001
culture_cons  1.744  0.253  0.000


## 8. Summary table for the report

In [12]:
summary = pd.DataFrame({
    'MIP = immigration (%)': 100*mip['immigration'],
    'MIP = economy/prices (%)': 100*mip['economy_prices'],
    'MIP = democracy/govt (%)': 100*mip['democracy_govt'],
    'immig > cost of living (%)': 100*hh['illegal immigration more important'],
    'Reps better on prices (%)': 100*own[own.issue == 'cost_living'].set_index('group')['Republicans'],
    'Reps better on immigration (%)': 100*own[own.issue == 'illegal_immig'].set_index('group')['Republicans'],
    'anti_elite': att['anti_elite'], 'econ_left': att['econ_left'], 'immig_cons': att['immig_cons'], 'culture_cons': att['culture_cons'],
    'N': att['N']}).loc[ORDER]
print(summary.round(2))
summary.to_csv(TABLES_DIR / 'anes_summary_by_group.csv')
print('\nSaved 10 tables to outputs/tables/ (anes_*.csv)')

               MIP = immigration (%)  MIP = economy/prices (%)  \
Biden->Harris                   3.31                     17.74   
Biden->Trump                   16.16                     34.79   
New->Harris                     5.77                     21.53   
New->Trump                     19.64                     34.49   
Trump->Harris                   7.56                     26.68   
Trump->Trump                   30.95                     31.09   

               MIP = democracy/govt (%)  immig > cost of living (%)  \
Biden->Harris                     30.21                        6.16   
Biden->Trump                       6.19                        4.52   
New->Harris                       15.98                        7.19   
New->Trump                         7.52                       14.61   
Trump->Harris                     18.58                       17.21   
Trump->Trump                      10.45                       13.19   

               Reps better on prices (%

## Limitations

- **N.** Sixty-nine Biden→Trump switchers and ~180 new voters per side. Every switcher percentage carries a ±10-point interval; the logit coefficients are indicative only. The CES (nb 11-13) remains the primary source for switcher *positions*; ANES adds salience and anti-elite items the CES does not ask.
- **ANES over-represents Harris voters** (two-party 53/47 vs certified 49/51). Group comparisons are the object; levels are not.
- **Post-election measurement.** MIP, importance, and ownership are all asked after the vote, so they are rationalizations as much as causes. The 2016-20-24 panel subsample (2,171) could in principle supply 2020-measured attitudes, but the 2020 ANES file is not on disk; see nb 12/13 for the CES panel version of the timing test.
- **MIP coding families are mine** (`MIP_FAMILY` in `anes_pull.py`); the "economy/prices" family pools economy, inflation, recession, energy prices; "democracy/govt" pools corruption, politicians, partisanship, elections, unity/division, and corporate corruption. Reassigning borderline codes (e.g. 32 race relations, 37 justice) does not change the immigration-vs-prices comparison.
- **Anti-elite index mixes pre-election and post-election items;** all respondents in the table have both interviews, so no bias from the mix, but the CSES items are late in the post questionnaire and have more breakoff missingness.